# 08 · Retries, poison records, and a dead-letter queue

**Level:** advanced · **Time:** 45–60 minutes · **Prerequisites:** notebooks 03–06.

You will distinguish invalid input from a temporary dependency failure, retry a small
bounded number of times, route a permanent failure to a dead-letter topic (DLQ), and
acknowledge the source **only after** the output is durably acknowledged by Kafka.

Start Kafka, restart the kernel, then run every cell in order. This notebook injects
failures intentionally; the expected failures are caught and explained.


In [ ]:
from pathlib import Path
import sys
from uuid import uuid4

ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
             if (p / "pyproject.toml").exists() and (p / "labtools").is_dir()), None)
if ROOT is None:
    raise RuntimeError("Open Jupyter from the kafka-django workshop folder.")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from labtools.kafka import (
    bootstrap_servers, client_config, require_broker, new_topic,
    producer, consumer, publish, poll_records, collect, decode,
)
RUN = uuid4().hex[:8]
print("Workshop:", ROOT.name, "| run:", RUN)


In [ ]:
import json
import time
from collections import Counter
from datetime import datetime, timezone
from confluent_kafka import TopicPartition

require_broker()
source_topic = new_topic("08.orders", partitions=2)
accepted_topic = new_topic("08.accepted", partitions=2)
dlq_topic = new_topic("08.dlq", partitions=2)
replay_topic = new_topic("08.replay", partitions=2)


## 1. Prepare four distinct failure cases

Money is integer paise to keep arithmetic exact. `temporary` will succeed on its third
attempt; `poison` never succeeds; `invalid` must not be retried because retrying cannot
change its invalid data. The fault names are teaching controls, not event schema fields.


In [ ]:
examples = [
    {"event_id": "normal", "customer_id": "c1", "amount_paise": 5000},
    {"event_id": "temporary", "customer_id": "c2", "amount_paise": 7000},
    {"event_id": "invalid", "customer_id": "c3", "amount_paise": -10},
    {"event_id": "poison", "customer_id": "c4", "amount_paise": 9000},
]
publish(source_topic, examples, key_field="customer_id")


In [ ]:
class TemporaryDependencyError(Exception):
    pass

def validate_order(value):
    if not isinstance(value, dict):
        raise ValueError("Order must be an object")
    for name in ("event_id", "customer_id"):
        if not isinstance(value.get(name), str) or not value[name]:
            raise ValueError(f"{name} must be a nonempty string")
    amount = value.get("amount_paise")
    if type(amount) is not int or amount <= 0:
        raise ValueError("amount_paise must be a positive integer")
    return value


## 2. Make the retry policy explicit

Only temporary failures get a retry. Three attempts means the initial call plus at
most two retries. We use very short delays for a live demo. Real policies need a time
budget, jitter, observability, and a decision about ordering while a partition is blocked.


In [ ]:
attempts = Counter()

def reserve_inventory(order):
    event_id = order["event_id"]
    attempts[event_id] += 1
    if event_id == "poison" or (event_id == "temporary" and attempts[event_id] < 3):
        raise TemporaryDependencyError("Injected inventory service failure")
    return {**order, "status": "reserved"}

def reserve_with_retry(order, max_attempts=3):
    for attempt in range(1, max_attempts + 1):
        try:
            return reserve_inventory(order)
        except TemporaryDependencyError:
            if attempt == max_attempts:
                raise
            time.sleep(0.05 * 2 ** (attempt - 1))


## 3. Preserve enough context to investigate a DLQ record

The envelope records the original topic, partition, offset, key, raw bytes, error class,
attempts, and a timestamp. Base64 preserves even input that is not valid UTF-8/JSON.
Store only data you are permitted to retain; a DLQ is still a data store with access and
retention requirements.


In [ ]:
import base64

def dead_letter(message, error, attempt_count):
    raw = message.value()
    return {
        "dlq_id": f"{message.topic()}:{message.partition()}:{message.offset()}",
        "source": {"topic": message.topic(), "partition": message.partition(),
                   "offset": message.offset()},
        "key_base64": (base64.b64encode(message.key()).decode()
                       if message.key() is not None else None),
        "value_base64": (base64.b64encode(raw).decode() if raw is not None else None),
        "error_type": type(error).__name__, "error": str(error),
        "attempts": attempt_count,
        "failed_at": datetime.now(timezone.utc).isoformat(),
    }


## 4. Publish the outcome before committing the source

`publish` checks delivery callbacks and a bounded flush; it raises if delivery is not
confirmed. If either output publish or source commit fails, this worker stops. It does
not continue and commit a later offset from the same partition over a failed record.
Processing one record at a time makes this rule visible.


In [ ]:
worker_group = f"lab-08-worker-{uuid4().hex}"
c = consumer(worker_group)
outcomes = []
try:
    c.subscribe([source_topic])
    for _ in range(len(examples)):
        message = poll_records(c, count=1, timeout=20)[0]
        value = None
        try:
            value = validate_order(decode(message))
            result = reserve_with_retry(value)
        except (ValueError, TypeError, UnicodeDecodeError, TemporaryDependencyError) as error:
            event_id = value.get("event_id", "") if isinstance(value, dict) else ""
            envelope = dead_letter(message, error, attempts[event_id])
            publish(dlq_topic, [envelope], key_field="dlq_id")
            destination = "dlq"
        else:
            publish(accepted_topic, [result], key_field="event_id")
            destination = "accepted"
        # Only reached after the output acknowledgement succeeds.
        c.commit(message=message, asynchronous=False)
        outcomes.append((message.partition(), message.offset(), destination))
finally:
    c.close()
print(outcomes)
print("Dependency attempts:", dict(attempts))
assert attempts == Counter(normal=1, temporary=3, poison=3)


In [ ]:
accepted = [decode(m) for m in collect(accepted_topic, count=2)]
failures = [decode(m) for m in collect(dlq_topic, count=2)]
assert {x["event_id"] for x in accepted} == {"normal", "temporary"}
assert {x["error_type"] for x in failures} == {"ValueError", "TemporaryDependencyError"}
for failure in failures:
    print(failure["source"], failure["error_type"], "attempts:", failure["attempts"])


**Observe:** invalid data bypassed the dependency, a transient fault recovered, and a
repeatedly failing valid record reached the DLQ. Moving it aside allows later source
records to progress, but can violate the business order of related events. Decide whether
blocking the partition, retry topics, or a per-entity pause is appropriate for your domain.


## 5. Prove that an unacknowledged DLQ write must not advance the source

Use a separate one-record topic and inject failure at the publish boundary. There is
deliberately no source commit in the exception branch. A restarted consumer in the same
group must receive the same input offset again.


In [ ]:
failure_topic = new_topic("08.publish-failure", partitions=1)
publish(failure_topic, [{"event_id": "retry-me", "amount_paise": -1}])
failure_group = f"lab-08-failure-{uuid4().hex}"
before = consumer(failure_group)
try:
    before.subscribe([failure_topic])
    original = poll_records(before, count=1, timeout=20)[0]
    try:
        raise RuntimeError("Injected: DLQ delivery was not acknowledged")
        # A commit here would lose the only durable copy of the failure outcome.
    except RuntimeError as error:
        print(error)
finally:
    before.close()

after = consumer(failure_group)
try:
    after.subscribe([failure_topic])
    replayed = poll_records(after, count=1, timeout=20)[0]
    assert replayed.offset() == original.offset()
    print("Replayed source offset:", replayed.offset())
finally:
    after.close()


## 6. Repair and replay deliberately

A DLQ is not a retry scheduler. Inspect a record, correct the cause, and send the approved
repair to a replay topic. We preserve the business event ID and add repair provenance.
A second replay may still produce duplicates, so the downstream service must deduplicate
durably or define a versioned correction event. This cell handles only the invalid amount.


In [ ]:
invalid_failure = next(x for x in failures if x["error_type"] == "ValueError")
repaired = json.loads(base64.b64decode(invalid_failure["value_base64"]))
repaired["amount_paise"] = 1000
repaired["repair_of"] = invalid_failure["dlq_id"]
validate_order(repaired)
publish(replay_topic, [repaired], key_field="event_id")
repaired_read = decode(collect(replay_topic, count=1)[0])
assert repaired_read["event_id"] == "invalid"
assert repaired_read["amount_paise"] == 1000
print(repaired_read)


## 7. Discuss the remaining crash gap

If output delivery succeeds and the process crashes before the source commit, the
source replays and the output may be published twice. This is **at-least-once** routing.
A Kafka transaction can join the output and source offsets (notebook 07). A database
side effect instead needs a durable event-ID record in the same database transaction
as that effect (notebook 12). This notebook makes no exactly-once claim.

Inline retries also consume the consumer's polling time budget. Long delays can trigger
rebalances. Retrying on separate topics permits progress but adds ordering, scheduling,
and operational decisions; sleeping for minutes in a consumer is not a robust scheduler.


## Practice

1. Add malformed UTF-8 bytes to the input and confirm the DLQ preserves the bytes.
2. Make `temporary` fail on four attempts. Predict the outcome with three allowed attempts.
3. Inject a failure immediately after output acknowledgement, then restart the group.
4. Track DLQ counts by error class and expose the age of the oldest unresolved failure.

**Solution hints:** use raw `producer.produce` for malformed bytes; it reaches the DLQ;
a post-publish crash can duplicate output; alert on age and rate, not just a nonempty topic.


## Presenter checkpoint and reference

Ask: which exact statement makes a source offset safe to commit? The intended answer
is that processing or the chosen durable failure handoff has succeeded, with duplicate
handling for the gap before the commit.

[Python consumer delivery semantics](https://docs.confluent.io/kafka-clients/python/current/overview.html#delivery-guarantees).
